In [1]:
import glob
from Bio import SeqIO 
import pandas as pd

In [83]:
files = glob.glob('../final_alignments/*.fa') + glob.glob('../final_alignments/extended-families/*.fa')

family = []
id = []
seq = []

for file in files:
    fam = file.split('/')[-1].replace('.fa', '')
    for record in SeqIO.parse(file, 'fasta'):
        family.append(fam)
        id.append(str(record.id).split('|')[0].replace('UniRef50_', '').replace('UniRef100_', '').split('/')[0])
        seq.append(str(record.seq).replace('-', ''))

/opt/miniconda3/lib/python3.13/site-packages/Bio/SeqIO/FastaIO.py:203: BiopythonDeprecationWarning: Previously, the FASTA parser silently ignored comments at the beginning of the FASTA file (before the first sequence).

Nowadays, the FASTA file format is usually understood not to have any such comments, and most software packages do not allow them. Therefore, the use of comments at the beginning of a FASTA file is now deprecated in Biopython.

In a future Biopython release, this deprecation warning will be replaced by a ValueError. To avoid this, there are three options:

(1) Modify your FASTA file to remove such comments at the beginning of the file.

(2) Use SeqIO.parse with the 'fasta-pearson' format instead of 'fasta'. This format is consistent with the FASTA format defined by William Pearson's FASTA aligner software. Thie format allows for comments before the first sequence; lines starting with the ';' character anywhere in the file are also regarded as comment lines and are ignor

In [4]:
len(id)

16728

In [84]:
df = pd.DataFrame({
    'family': family,
    'id': id,
    'seq': seq
})

In [69]:
df.head()

,family,id,seq
0,BrnT,A0A6I3B2E8,MSENFEFDDDKNRSNRIKHGITFGFARGIWLDRNMVVLQAKTEQEE...
1,BrnT,A0A2D5JIB4,MKFEWNEDKNTLNKQKHGISFEEAKEIFDDALHLSKLDNYELEDNY...
2,BrnT,A0A161VN93,MVFKWDESKAAINLKKHNVSFEEAKTVFDKGATRFCEMTLLKERSL...
3,BrnT,A0A496WXS3,MALDFEWDQIKSRQNTKKHGISFREGATAFADKLSYTISDPEHSHA...
4,BrnT,A0A551Y2G8,VKLAREFDWDANNIEHISRHNLIPAEVESVFQDTRKIGTASRKTER...


In [85]:
from IPython.display import HTML

HTML(df.groupby('family').nunique().to_html())


,id,seq
family,,
BC_0920,205,205
BECR-Tox1,28,28
BECR-Tox2,11,11
BECR-Tox3,12,12
BECR-Tox4,30,30
BECR-Tox5,61,61
BECR-Tox6.2,7,7
Barnase,69,69
BrnT,1078,1078


In [86]:
df = df[~df['family'].isin(['Sarcin_full', 'CdiA-CT_EC3006_full', 'MafB'])]

In [89]:
# Step 1: get list of families per seq
seq_families = (
    df.groupby('seq')['family']
      .unique()
      .reset_index(name='families')
)

# Step 2: keep only sequences that appear in multiple families
multi_seq_families = seq_families[seq_families['families'].str.len() > 1]

# Step 3: merge back to get id, seq, and families
result = (
    df.merge(multi_seq_families[['seq', 'families']], on='seq')
      .sort_values(['seq', 'id'])
)

result

,family,id,seq,families
5,Ntox49.4,M5DVV8,FLDHALDRMEQRGVTTRQVLRVLRDGDQEGNAEWCTDKERGWRCRL...,"[CdiA-CT_96.154, Ntox49.4]"
3,CdiA-CT_96.154,M5DVV8_9GAMM,FLDHALDRMEQRGVTTRQVLRVLRDGDQEGNAEWCTDKERGWRCRL...,"[CdiA-CT_96.154, Ntox49.4]"
4,Ntox49.3,A0A520XF66,FSLHALSMAQERDISHEWIRKTIENPDYTELREDGTMHYISAIPER...,"[CdiA-CT_96.154, Ntox49.3]"
2,CdiA-CT_96.154,Q2FU07_METHJ,FSLHALSMAQERDISHEWIRKTIENPDYTELREDGTMHYISAIPER...,"[CdiA-CT_96.154, Ntox49.3]"
1,ParE,E6X208,MKLSRSRQFKKDLRNYATQMGDKHFQSLIEALSCLMEGKPLPSYCR...,"[YafQ, ParE]"
0,YafQ,E6X208_NITSE,MKLSRSRQFKKDLRNYATQMGDKHFQSLIEALSCLMEGKPLPSYCR...,"[YafQ, ParE]"


In [90]:
HTML(result.to_html())

,family,id,seq,families
5,Ntox49.4,M5DVV8,FLDHALDRMEQRGVTTRQVLRVLRDGDQEGNAEWCTDKERGWRCRLSRITAGEKITVIAKLVERKNSTCLVVTTWE,"[CdiA-CT_96.154, Ntox49.4]"
3,CdiA-CT_96.154,M5DVV8_9GAMM,FLDHALDRMEQRGVTTRQVLRVLRDGDQEGNAEWCTDKERGWRCRLSRITAGEKITVIAKLVERKNSTCLVVTTWE,"[CdiA-CT_96.154, Ntox49.4]"
4,Ntox49.3,A0A520XF66,FSLHALSMAQERDISHEWIRKTIENPDYTELREDGTMHYISAIPERKGKFLRVIINPSVQPQRIITVF,"[CdiA-CT_96.154, Ntox49.3]"
2,CdiA-CT_96.154,Q2FU07_METHJ,FSLHALSMAQERDISHEWIRKTIENPDYTELREDGTMHYISAIPERKGKFLRVIINPSVQPQRIITVF,"[CdiA-CT_96.154, Ntox49.3]"
1,ParE,E6X208,MKLSRSRQFKKDLRNYATQMGDKHFQSLIEALSCLMEGKPLPSYCRDHPLKGRLAGYRELHIGGDLLVLYTIEDEVIYLIRLGSHAEILG,"[YafQ, ParE]"
0,YafQ,E6X208_NITSE,MKLSRSRQFKKDLRNYATQMGDKHFQSLIEALSCLMEGKPLPSYCRDHPLKGRLAGYRELHIGGDLLVLYTIEDEVIYLIRLGSHAEILG,"[YafQ, ParE]"


In [91]:
import Levenshtein
import pandas as pd

seqs = df['seq'].unique()

clusters = []
used = set()

for s in seqs:
    if s in used:
        continue
    cluster = [s]
    used.add(s)
    for t in seqs:
        if t in used:
            continue
        if Levenshtein.distance(s, t) <= 3:   # threshold for “similar”
            cluster.append(t)
            used.add(t)
    clusters.append(cluster)

seq_to_cluster = {}
for i, clust in enumerate(clusters):
    for s in clust:
        seq_to_cluster[s] = i

df['cluster'] = df['seq'].map(seq_to_cluster)

result = (
    df.groupby('cluster')
      .agg(
          ids=('id', list),
          seqs=('seq', list),
          families=('family', lambda x: list(set(x)))
      )
      .query('families.str.len() > 1')
      .reset_index()
)


In [93]:
from Bio import pairwise2

def seq_identity(a, b):
    score = pairwise2.align.globalxx(a, b, score_only=True)
    return score / max(len(a), len(b))

seqs = df['seq'].unique()
clusters = []
used = set()

threshold = 0.95

for s in seqs:
    if s in used:
        continue
    cluster = [s]
    used.add(s)
    for t in seqs:
        if t in used:
            continue
        if seq_identity(s, t) >= threshold:
            cluster.append(t)
            used.add(t)
    clusters.append(cluster)

    seq_to_cluster = {}
for i, clust in enumerate(clusters):
    for s in clust:
        seq_to_cluster[s] = i

df['cluster'] = df['seq'].map(seq_to_cluster)

result = (
    df.groupby('cluster')
      .agg(
          ids=('id', list),
          seqs=('seq', list),
          families=('family', lambda x: list(set(x)))
      )
      .query('families.str.len() > 1')
      .reset_index()
)

result



/opt/miniconda3/lib/python3.13/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


KeyboardInterrupt: 

In [92]:
result

,cluster,ids,seqs,families
0,3222,"[E5BI39_9FUSO, UPI000219C0ED]",[KVQGIFEWFKQFHGKTIRIEGESKNFEILLDNESLPHLLGVQYVN...,"[PBECR4, PBECR4.2]"
1,3256,"[U3PBN3_9CAUD, U3PBN3]",[DLQNILNDFINCFCNGYVEIKTKYKILPIFKISFHKNNLPHLLGL...,"[PBECR4, PBECR4.2]"
2,3466,"[WP_099151663.1, WP_099151663.1]",[NELPAYENPGHHDPSTNSYISNKSVLPENHEELFKNSVPDPNNPK...,"[Ntox18.st2, Ntox18.st1]"
3,3468,"[NVJ50159.1, NVJ50159.1]",[DPDEDDEYENPGHHDPHLRGHNTYNKTKSVLPKDHESLWRSSRVA...,"[Ntox18.st2, Ntox18.st1]"
4,3470,"[GFN06613.1, GFN06613.1]",[HNSTCSNYENPGHHDPTGGPNPYVPKRAVLPGDAGEQFGNSVLVD...,"[Ntox18.st2, Ntox18.st1]"
...,...,...,...,...
74,11543,"[Q2FU07_METHJ, A0A520XF66]",[FSLHALSMAQERDISHEWIRKTIENPDYTELREDGTMHYISAIPE...,"[CdiA-CT_96.154, Ntox49.3]"
75,11545,"[Q8FSY0_COREF, UPI001E4C9830]",[ISPHARARMEQRGVTEQEIVETLSNPDLTRPGNKPDRTIYERNIG...,"[CdiA-CT_96.154, Ntox49.3]"
76,11546,"[G0EFM2_PYRF1, A0A933PSX6]",[LTRHAEERLRERGISLEEVKVCIQNPDYTEESRGALRCWKRMGDK...,"[CdiA-CT_96.154, Ntox49.3]"
77,11550,"[M5DVV8_9GAMM, M5DVV8]",[FLDHALDRMEQRGVTTRQVLRVLRDGDQEGNAEWCTDKERGWRCR...,"[CdiA-CT_96.154, Ntox49.4]"
